In [6]:
import json,pickle,zipfile,re,math,os,time
import numpy as np
import unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from peft import LoraConfig,AdaLoraConfig, get_peft_model
from collections import Counter
from tqdm import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader
from torch.utils import checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [8]:
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
## tokenizer
 
WORD_END = 256
 
def pre_tokenize_preserve(text):
    return re.findall(r"\w+|[^\w\s]|\s+", text)
 
def word_to_symbols(word):
    byte_values = word.encode("utf-8")
    symbols = [(b,) for b in byte_values]
    symbols.append((WORD_END,))
    return tuple(symbols)
 
def get_adjacent_pairs(symbols):
    return set(zip(symbols, symbols[1:]))
 
def apply_bpe_merges(symbols, merge_rank):
    symbols = list(symbols)
    while len(symbols) > 1:
        pairs = get_adjacent_pairs(symbols)
        candidates = [p for p in pairs if p in merge_rank]
        if not candidates:
            break
        best_pair = min(candidates, key=lambda p: merge_rank[p])
        sa, sb = best_pair
        merged = sa + sb
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == sa and symbols[i + 1] == sb:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols
 
def load_tokenizer(pkl_path):
    bpe_data = pickle.load(open(pkl_path, "rb"))
    stoi, merge_rank = bpe_data["stoi"], bpe_data["merge_rank"]
    cache = {}
 
    def encode_unit(unit):
        if unit not in cache:
            cache[unit] = [stoi[s] for s in apply_bpe_merges(word_to_symbols(unit), merge_rank)]
        return cache[unit]
 
    def encode(text):
        ids = []
        for unit in pre_tokenize_preserve(text):
            ids += encode_unit(" " if unit.isspace() else unit)
        return ids
 
    return bpe_data, encode
 
 
def clean_text(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"(\w)-[ \t]*\n[ \t]*(\w)", r"\1\2", t)
    return re.sub(r"\s+", " ", t).strip()

# Model V2

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RoPE(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Model V5

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present

class Expert(nn.Module):
    def __init__(self, n_embd, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden = expert_hidden_mult * n_embd
        self.c_fc = nn.Linear(n_embd, hidden)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class MoEFeedForward(nn.Module):
    def __init__(self, n_embd, num_experts=4, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(n_embd, expert_hidden_mult, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)                                  

        router_logits = self.router(x_flat)                     
        router_probs = F.softmax(router_logits, dim=-1)
        top1_prob, top1_idx = router_probs.max(dim=-1)          
        one_hot = F.one_hot(top1_idx, num_classes=self.num_experts).to(x.dtype)   

        expert_outputs = torch.stack([expert(x_flat) for expert in self.experts], dim=1)  
        gate = (one_hot * top1_prob.unsqueeze(-1)).unsqueeze(-1)                          
        output = (expert_outputs * gate).sum(dim=1)                                        
        with torch.no_grad():
            tokens_per_expert = one_hot.float().mean(dim=0)      
        mean_router_prob = router_probs.mean(dim=0)              
        aux_loss = self.num_experts * (tokens_per_expert * mean_router_prob).sum()

        return output.view(B, T, C), aux_loss

class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, num_experts, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_1_post = RMSNorm(n_embd)

        self.ln_2_pre = RMSNorm(n_embd)
        self.moe = MoEFeedForward(n_embd, num_experts=num_experts, dropout=dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward_train(self, x, attn_mask):
        attn_out, _ = self.attn(self.ln_1_pre(x), attn_mask)
        x = x + self.ln_1_post(attn_out)

        moe_out, aux_loss = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, aux_loss

    def forward_infer(self, x, past_kv, use_cache, position_offset):
        attn_out, present = self.attn(self.ln_1_pre(x), None, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)

        moe_out, _ = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, present

class DecoderOnlyLM_MoE_Sandwich(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,num_experts=4, aux_loss_weight=0.01, dropout=0.1, pad_id=0,gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.aux_loss_weight = aux_loss_weight
        self.gradient_checkpointing = gradient_checkpointing

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, num_experts, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        x = self.drop(self.wte(input_ids))

        if use_cache:
            presents = []
            for i, block in enumerate(self.blocks):
                past = past_key_values[i] if past_key_values is not None else None
                x, present = block.forward_infer(x, past, use_cache, past_len)
                presents.append(present)
        else:
            combined_mask = None
            if attention_mask is not None:
                key_mask = attention_mask.bool()[:, None, None, :]
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]

            aux_losses = []
            for block in self.blocks:
                if self.gradient_checkpointing and self.training:
                    x, aux = checkpoint.checkpoint(block.forward_train, x, combined_mask,
                                                   use_reentrant=False)
                else:
                    x, aux = block.forward_train(x, combined_mask)
                aux_losses.append(aux)
            presents = None

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            ce_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
            aux_loss = torch.stack(aux_losses).mean() if not use_cache else torch.tensor(0.0, device=x.device)
            loss = ce_loss + self.aux_loss_weight * aux_loss

        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=None,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_active_parameters(model, num_experts):
    total = count_parameters(model)
    expert_params = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
    num_layers = len(model.blocks)
    unused_experts_params = expert_params * (num_experts - 1) * num_layers
    return total - unused_experts_params

# Model V6

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def is_global_layer(layer_idx, n_layer, global_every_n):
    return (n_layer - 1 - layer_idx) % global_every_n == 0


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size=None, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head
        self.window_size = window_size   

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def _build_mask(self, T, device, key_padding_mask=None):
        i = torch.arange(T, device=device)
        if self.window_size is None:
            causal = i[:, None] >= i[None, :]                              
        else:
            diff = i[:, None] - i[None, :]
            causal = (diff >= 0) & (diff < self.window_size)               
        mask = causal[None, None, :, :]
        if key_padding_mask is not None:
            key_mask = key_padding_mask.bool()[:, None, None, :]
            eye = torch.eye(T, dtype=torch.bool, device=device)
            mask = (mask & key_mask) | eye[None, None, :, :]
        return mask

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
            if self.window_size is not None and k.size(2) > self.window_size:
                k = k[:, :, -self.window_size:, :]
                v = v[:, :, -self.window_size:, :]
                if key_padding_mask is not None:
                    key_padding_mask = key_padding_mask[:, -self.window_size:]
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if past_kv is None and T > 1:
            attn_mask = self._build_mask(T, x.device, key_padding_mask)
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        else:
            attn_mask = None
            if key_padding_mask is not None:
                attn_mask = key_padding_mask.bool()[:, None, None, :]
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, window_size, dropout)
        self.ln_1_post = RMSNorm(n_embd)
        self.ln_2_pre = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1_pre(x), key_padding_mask, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)
        x = x + self.ln_2_post(self.mlp(self.ln_2_pre(x)))
        return (x, present) if use_cache else x

class DecoderOnlyLM_HybridMask(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                window_size=192, global_every_n=4, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.window_size = window_size
        self.global_every_n = global_every_n

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)

        layer_windows = [
            None if is_global_layer(i, n_layer, global_every_n) else window_size
            for i in range(n_layer)
        ]
        print(f"Layer attention pattern (None = global/full-causal): {layer_windows}")

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, layer_windows[i], dropout)
            for i in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        if attention_mask is None and past_key_values is None:
            attention_mask = input_ids.ne(self.pad_id)

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, attention_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, attention_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]
        running_mask = attention_mask

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            running_mask = torch.cat([running_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=running_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Model V7

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RMSNorm_GQA(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
def build_model():
    # for  v7
    return DecoderOnlyLM_RMSNorm_GQA(
    vocab_size=VOCAB_SIZE,         
    context_length=1536,
    n_embd=768, 
    n_layer=10,
    n_head=12, 
    n_kv_head=3,
    pad_id=PAD_ID,                 
    gradient_checkpointing=True,
) 
# for v2
#       return DecoderOnlyLM_RoPE(
#     vocab_size=VOCAB_SIZE,          
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v5
#       return DecoderOnlyLM_MoE_Sandwich(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     num_experts=2,
#     aux_loss_weight=0.01,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v6
#     return DecoderOnlyLM_HybridMask(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     window_size=192,
#     global_every_n=4,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )


HEAD_PATH   = "lm_head"      
LAYERS_PATH = "blocks"           
FINAL_NORM  = ("ln_f",)      
MODEL_PT    = "/kaggle/working/decoder_baseline_v7_final_model.pt"

In [ ]:
def make_offset_encoder(bpe_data, pre_tokenize_preserve, word_to_symbols,apply_bpe_merges, word_end, canonicalize_whitespace=True):
    stoi = bpe_data["stoi"]
    merge_rank = bpe_data["merge_rank"]
    cache = {}

    def unit_tokens(unit):
        if unit in cache:
            return cache[unit]
        symbols = apply_bpe_merges(word_to_symbols(unit), merge_rank)
        char_of_byte = []
        for i, ch in enumerate(unit):
            char_of_byte.extend([i] * len(ch.encode("utf-8")))
        toks, byte_pos = [], 0
        for s in symbols:
            n = sum(1 for b in s if b != word_end)
            if n == 0:                       
                cs = ce = len(unit)
            else:
                cs = char_of_byte[byte_pos]
                ce = char_of_byte[byte_pos + n - 1] + 1
            byte_pos += n
            toks.append((stoi[s], cs, ce))
        cache[unit] = toks
        return toks

    def encode_with_offsets(text):
        ids, offsets, pos = [], [], 0
        for unit in pre_tokenize_preserve(text):
            u_start, u_end = pos, pos + len(unit)
            pos = u_end
            if canonicalize_whitespace and unit.isspace():
                for tid, _, _ in unit_tokens(" "):
                    ids.append(tid)
                    offsets.append((u_start, u_end))
            else:
                for tid, cs, ce in unit_tokens(unit):
                    ids.append(tid)
                    offsets.append((u_start + cs, u_start + ce))
        return ids, offsets

    return encode_with_offsets

def normalize_gold(raw_gold, texts, verbose=True):
    out, mismatches, total = {}, 0, 0
    for doc_id, ents in raw_gold.items():
        spans = []
        for x in ents:
            if isinstance(x, dict):
                s, e, lab, et = x["start"], x["end"], x["label"], x.get("text")
            else:
                s, e, lab = x[0], x[1], x[2]
                et = x[3] if len(x) > 3 else None
            s, e = int(s), int(e)
            total += 1
            if et is not None and doc_id in texts and texts[doc_id][s:e] != et:
                mismatches += 1
            spans.append((s, e, lab))
        out[doc_id] = sorted(spans)
    if verbose:
        print(f"gold entities: {total}; text[start:end] != entity_text for {mismatches}")
    return out, mismatches

def find_overlaps(gold):
    n = 0
    for spans in gold.values():
        spans = sorted(spans)
        for a, b in zip(spans, spans[1:]):
            if b[0] < a[1]:
                n += 1
    return n

def build_tag_maps(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}

def spans_to_token_tags(offsets, spans):
    spans = sorted(spans)
    tags = ["O"] * len(offsets)
    j, started = 0, -1
    for i, (s, e) in enumerate(offsets):
        if e == s:
            if i > 0 and tags[i - 1] != "O":
                tags[i] = "I-" + tags[i - 1].split("-", 1)[1]
            continue
        while j < len(spans) and spans[j][1] <= s:
            j += 1
        if j >= len(spans):
            continue
        es, ee, lab = spans[j]
        overlap = min(e, ee) - max(s, es)
        if overlap > 0 and overlap * 2 >= (e - s):
            tags[i] = ("I-" if started == j else "B-") + lab
            started = j
    return tags


def token_tags_to_spans(text, offsets, tags):
    spans, cur = [], None

    def close():
        nonlocal cur
        if cur is not None:
            s, e, lab = cur
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if e > s:
                spans.append((s, e, lab))
            cur = None

    for (s, e), tag in zip(offsets, tags):
        if tag == "O":
            close()
            continue
        prefix, lab = tag.split("-", 1)
        if prefix == "B" or cur is None or cur[2] != lab:
            close()
            cur = [s, e, lab]
        else:
            cur[1] = max(cur[1], e)
    close()
    return spans

def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc in set(gold) | set(pred):
        g = {(int(a), int(b), l) for a, b, l, *_ in gold.get(doc, [])}
        p = {(int(a), int(b), l) for a, b, l, *_ in pred.get(doc, [])}
        for _, _, l in g & p:
            tp[l] += 1
        for _, _, l in p - g:
            fp[l] += 1
        for _, _, l in g - p:
            fn[l] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = {"precision": P, "recall": R, "f1": F, "support": tp[l] + fn[l]}
    scored = [l for l in labels if per[l]["support"] > 0] 
    macro = {k: sum(per[l][k] for l in scored) / max(1, len(scored))
             for k in ("precision", "recall", "f1")}
    return {"per_label": per, "macro": macro}


def oracle_check(texts, gold, encode_with_offsets, labels, max_missed=10):
    pred, missed = {}, []
    for doc_id, text in texts.items():
        if doc_id not in gold:
            continue
        _, offs = encode_with_offsets(text)
        tags = spans_to_token_tags(offs, gold[doc_id])
        pred[doc_id] = token_tags_to_spans(text, offs, tags)
        for s, e, l in set(gold[doc_id]) - set(pred[doc_id]):
            if len(missed) < max_missed:
                missed.append((doc_id, s, e, l, text[s:e]))
    gold_f = {d: gold[d] for d in pred}
    return strict_scores(gold_f, pred, labels), missed

def make_windows(n_tokens, max_len, stride):
    assert 0 < stride <= max_len
    if n_tokens <= max_len:
        return [(0, n_tokens)]
    starts = list(range(0, n_tokens - max_len + 1, stride))
    if starts[-1] + max_len < n_tokens:
        starts.append(n_tokens - max_len)
    return [(s, s + max_len) for s in starts]


def merge_window_scores(n_tokens, windows, window_scores):
    C = window_scores[0].shape[1]
    acc = np.zeros((n_tokens, C))
    wsum = np.zeros((n_tokens, 1))
    for (s, e), sc in zip(windows, window_scores):
        L = e - s
        w = np.minimum(np.arange(L) + 1, L - np.arange(L)).astype(float)[:, None]
        acc[s:e] += sc * w
        wsum[s:e] += w
    return acc / wsum


def make_training_examples(doc_id, ids, tag_ids, max_len, stride):
    return [{"doc_id": doc_id, "start": s, "input_ids": ids[s:e], "labels": tag_ids[s:e]} for s, e in make_windows(len(ids), max_len, stride)]

def collate_ner(batch, pad_id, ignore_index=-100):
    L = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), L), ignore_index, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.bool)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        attn[i, :n] = True
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attn}


In [ ]:
# NER Model
def load_phase1_weights(model, path):
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=True)
    return model

class NERModel(nn.Module):
    def __init__(self, backbone, num_tags, head_path="lm_head", dropout=0.1):
        super().__init__()
        self.backbone = backbone
        parent = backbone
        *parents, name = head_path.split(".")
        for p in parents:
            parent = getattr(parent, p)
        assert isinstance(getattr(parent, name), nn.Module), f"{head_path} not found"
        setattr(parent, name, nn.Identity())
        hidden = next(m for m in backbone.modules() if isinstance(m, nn.Embedding)).embedding_dim
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_tags)

    def forward(self, input_ids):
        out = self.backbone(input_ids)
        h = out["logits"] if isinstance(out, dict) else out
        return self.classifier(self.drop(h))

def get_layers(backbone, layers_path=None):
    if layers_path:
        m = backbone
        for part in layers_path.split("."):
            m = getattr(m, part)
        return m
    lists = [(n, m) for n, m in backbone.named_modules() if isinstance(m, nn.ModuleList)]
    top = max(len(m) for _, m in lists)
    cands = [(n, m) for n, m in lists if len(m) == top]
    return cands[0][1]


def set_trainable(ner_model, mode, n_last_layers=4, layers_path=None, top_level_unfreeze=("ln_f",)):
    assert mode in ("full", "partial")
    if mode == "full":
        for p in ner_model.parameters():
            p.requires_grad = True
    else:
        for p in ner_model.parameters():
            p.requires_grad = False
        layers = get_layers(ner_model.backbone, layers_path)
        for blk in list(layers)[-n_last_layers:]:
            for p in blk.parameters():
                p.requires_grad = True
        for name, p in ner_model.backbone.named_parameters():
            if any(part in top_level_unfreeze for part in name.split(".")):
                p.requires_grad = True
        for p in ner_model.classifier.parameters():
            p.requires_grad = True

    trainable = sum(p.numel() for p in ner_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in ner_model.parameters())
    print(f"[{mode}] trainable {trainable/1e6:.2f}M / {total/1e6:.2f}M "f"({100*trainable/total:.1f}%)")
    if mode == "partial":
        groups = sorted({".".join(n.split(".")[:-1]) for n, p in ner_model.named_parameters()
                         if p.requires_grad})
        print("trainable modules:", groups[:8], "..." if len(groups) > 8 else "")
    return trainable, total

def bio_transition_mask(tags):
    T = len(tags); ok = np.zeros((T, T), bool)
    for i, a in enumerate(tags):
        for j, b in enumerate(tags):
            if b == "O" or b.startswith("B-"): ok[i, j] = True
            else:                                  
                ok[i, j] = a != "O" and a.split("-", 1)[1] == b.split("-", 1)[1]
    start_ok = np.array([t == "O" or t.startswith("B-") for t in tags])
    return ok, start_ok
 
def viterbi_decode(logp, tags, o_bias=0.0):
    n, T = logp.shape
    if n == 0:
        return []
    ok, start_ok = bio_transition_mask(tags)
    NEG = -1e30
    trans = np.where(ok, 0.0, NEG)
    lp = logp.copy(); lp[:, tags.index("O")] += o_bias
    score = np.where(start_ok, lp[0], NEG)
    back = np.zeros((n, T), dtype=np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans                  
        back[t] = cand.argmax(0)
        score = cand.max(0) + lp[t]
    path = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return [tags[i] for i in path[::-1]]


@torch.no_grad()
def predict_docs(ner_model, texts, tok_docs, tags, pad_id, max_len, stride,device, batch_size=4, use_amp=True):
    ner_model.eval()
    amp = use_amp and device.type == "cuda"
    preds = {}
    for d, v in tok_docs.items():
        n = len(v["ids"])
        wins = make_windows(n, max_len, stride)
        scores = []
        for i in range(0, len(wins), batch_size):
            chunk = wins[i:i + batch_size]
            batch = [{"input_ids": v["ids"][s:e], "labels": [0] * (e - s)} for s, e in chunk]
            b = collate_ner(batch, pad_id)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = ner_model(b["input_ids"].to(device))
            logp = F.log_softmax(logits.float(), dim=-1).cpu().numpy()
            for j, (s, e) in enumerate(chunk):
                scores.append(logp[j, : e - s])
        merged = merge_window_scores(n, wins, scores)
        tag_seq = viterbi_decode(merged, tags, o_bias=0.0)
        preds[d] = token_tags_to_spans(texts[d], v["offs"], tag_seq)
    return preds


def show_linear_names(model):
    names = sorted({n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, nn.Linear)})
    print("Linear module name suffixes:", names)
    return names


def apply_lora(ner_model, r=8, alpha=16, dropout=0.1, target_modules=("q_proj", "v_proj")):
    from peft import LoraConfig, get_peft_model
    tm = target_modules if isinstance(target_modules, str) else list(target_modules)
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=tm,modules_to_save=["classifier"], bias="none")
    peft_model = get_peft_model(ner_model, cfg)
    peft_model.print_trainable_parameters()
    return peft_model

def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def train_ner(ner_model, train_examples, pad_id, cfg, device, val_fn=None, save_path=None):
    ner_model.to(device)
    head_ids = {id(p) for p in ner_model.classifier.parameters()}
    head_params = [p for p in ner_model.parameters() if id(p) in head_ids and p.requires_grad]
    backbone_params = [p for p in ner_model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

    loader = DataLoader(train_examples, batch_size=cfg["batch_size"], shuffle=True,collate_fn=lambda b: collate_ner(b, pad_id))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)

    best, history, step = -1.0, [], 0
    for epoch in range(cfg["epochs"]):
        ner_model.train()
        run_loss, n_batches = 0.0, 0
        opt.zero_grad()
        for i, b in enumerate(loader):
            x, y = b["input_ids"].to(device), b["labels"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = ner_model(x)
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1),ignore_index=-100)
            scaler.scale(loss / cfg["grad_accum"]).backward()
            if epoch == 0 and i == 0:
                dead = [n for n, p in ner_model.named_parameters()
                        if p.requires_grad and p.grad is None]
                if dead:
                    print(f"WARNING: {len(dead)} trainable params got no gradient")
            run_loss += loss.item()
            n_batches += 1
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(ner_model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad()
                step += 1
        rec = {"epoch": epoch + 1, "train_loss": run_loss / max(1, n_batches)}
        if val_fn is not None:
            rec["val"] = val_fn(ner_model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(ner_model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        ner_model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val={best:.4f})")
    return history

def write_submission(preds, json_path="prediction.json", zip_path="submission.zip"):
    out = {d: [[int(s), int(e), l] for s, e, l in spans] for d, spans in preds.items()}
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(out, f)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")
    print(f"wrote {zip_path}: {len(out)} docs, {sum(len(v) for v in out.values())} entities")


In [ ]:
_ROT_IDX = {}
 
def _rot_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                 
    return _ROT_IDX[key]
 
 
class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, variant="plus", dropout=0.1):
        super().__init__()
        assert variant in ("plus", "pi")
        self.in_features, self.out_features = base.in_features, base.out_features
        self.weight, self.bias = base.weight, base.bias          
        self.weight.requires_grad = False
        if self.bias is not None:
            self.bias.requires_grad = False
        d, dev = base.out_features, base.weight.device
        self.variant, self.r, self.scale = variant, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, d, device=dev))
        self.B = nn.Parameter(torch.empty(d, r, device=dev))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        with torch.no_grad():
            if variant == "plus":
                nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
                self.B.zero_()
            else:
                self.A.zero_()
                self.A[0, 0] = 1.0
                nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))   
                self.B[:, 0] = 1.0
 
    def forward(self, x):
        y = F.linear(x, self.weight)                              
        if self.variant == "plus":                                
            y = y + self.scale * F.linear(F.linear(self.drop(y), self.A), self.B)
        else:                                                   
            M = torch.gather(self.B @ self.A, 1, _rot_index(self.out_features, y.device))
            y = F.linear(y, M)
        return y if self.bias is None else y + self.bias
 
    @torch.no_grad()
    def merged_weight(self):
        d = self.out_features
        if self.variant == "plus":
            M = torch.eye(d, device=self.weight.device) + self.scale * (self.B @ self.A)
        else:
            M = torch.gather(self.B @ self.A, 1, _rot_index(d, self.weight.device))
        return M @ self.weight
 
 
def apply_lorma(ner, r=8, alpha=16, variant="plus", dropout=0.1, target_modules=("q_proj", "v_proj")):
    for p in ner.parameters():
        p.requires_grad = False
    targets, n = set(target_modules), 0
    for _, parent in list(ner.backbone.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(parent, cname, LoRMALinear(child, r, alpha, variant, dropout))
                n += 1
    assert n > 0, f"no Linear layers matched {targets}"
    for p in ner.classifier.parameters():
        p.requires_grad = True
    tr = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in ner.parameters())
    print(f"LoRMA-{variant}: adapted {n} Linear layers | trainable {tr/1e6:.3f}M / {tot/1e6:.2f}M")
    return ner
 
 
@torch.no_grad()
def merge_lorma(ner):
    for _, parent in list(ner.backbone.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, LoRMALinear):
                lin = nn.Linear(child.in_features, child.out_features,
                                bias=child.bias is not None).to(child.weight.device)
                lin.weight.copy_(child.merged_weight())
                if child.bias is not None:
                    lin.bias.copy_(child.bias)
                for q in lin.parameters():
                    q.requires_grad = False
                setattr(parent, cname, lin)
    return ner
 

In [ ]:
from peft import LoraConfig, get_peft_model

def snap_words(text, s, e):
    while s > 0 and text[s - 1].isalnum() and text[s].isalnum():
        s -= 1
    while e < len(text) and text[e - 1].isalnum() and text[e].isalnum():
        e += 1
    return s, e
 
def postprocess(preds, texts, gap=1, snap=True):
    out = {}
    for d, spans in preds.items():
        text, spans = texts[d], sorted(spans)
        if snap:
            spans = [(*snap_words(text, s, e), l) for s, e, l in spans]
        merged = []
        for s, e, l in spans:
            if merged and merged[-1][2] == l and s - merged[-1][1] <= gap:
                merged[-1] = (merged[-1][0], max(e, merged[-1][1]), l)
            else:
                merged.append((s, e, l))
        out[d] = merged
    return out
 
def viterbi_spans(merged_docs, texts, tok_docs, tags, o_bias=0.0):
    return {d: token_tags_to_spans(texts[d], tok_docs[d]["offs"], viterbi_decode(m, tags, o_bias=o_bias))for d, m in merged_docs.items()}
 
def _train_v2(ner, cfg, tag, decode_kw=None):
    decode_kw = decode_kw or dict(collapse=True, gap=1, o_bias=0.5, snap=True)
    def val_fn(m):
        merged = get_merged_scores(m, tok2, PAD_ID, MAX_LEN, STRIDE, device)
        pr = decode(merged, texts2, tok2, tags, labels, **decode_kw)
        return strict_scores(gold2, pr, labels)["macro"]["f1"]
 
    hist = train_ner(ner, train_ex, PAD_ID, cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/ner_{tag}_best.pt")
    return ner, hist
 
def apply_lora_v2(ner_model, r=32, alpha=64, dropout=0.1,target_modules=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),use_rslora=False, use_dora=False, extra_modules_to_save=("ln_f",)):
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=list(target_modules),
                     modules_to_save=["classifier", *extra_modules_to_save], bias="none",
                     use_rslora=use_rslora, use_dora=use_dora)
    pm = get_peft_model(ner_model, cfg)
    pm.print_trainable_parameters()
    return pm

class BiLSTMHead(nn.Module):
    def __init__(self, hidden, num_tags, lstm_hidden=256, dropout=0.1,num_layers=1):
        super().__init__()
        self.in_features = hidden
        self.lstm=nn.LSTM(hidden, lstm_hidden, num_layers=num_layers, batch_first=True, bidirectional=True,dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * lstm_hidden, num_tags)
 
    def forward(self, h, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(h, lengths.cpu(), batch_first=True, enforce_sorted=False)
        o, _ = self.lstm(packed)
        o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=h.size(1))
        return self.out(self.drop(o))
 
 
class NERModelBi(NERModel):
    def __init__(self, backbone, num_tags, pad_id, head_path="lm_head", dropout=0.1, lstm_hidden=256,lstm_layers=1):
        super().__init__(backbone, num_tags, head_path, dropout)
        hidden = self.classifier.in_features
        self.classifier = BiLSTMHead(hidden, num_tags, lstm_hidden, dropout,lstm_layers)
        self.pad_id = pad_id
 
    def forward(self, input_ids):
        out = self.backbone(input_ids)
        h = out["logits"] if isinstance(out, dict) else out
        if h.size(1) == 0:                                    
            return h.new_zeros(h.size(0), 0, len(tags))
        lengths = (input_ids != self.pad_id).sum(1).clamp(min=1)
        return self.classifier(self.drop(h), lengths)

def ensemble_merged(models, tok_docs):
    ms = [get_merged_scores(m, tok_docs, PAD_ID, MAX_LEN, STRIDE, device) for m in models]
    return {d: np.mean([m[d] for m in ms], axis=0) for d in ms[0]}

In [ ]:
#knowledge distillation
def windows_from_tok(tok_docs, with_labels=False):
    ex = []
    for d, v in tok_docs.items():
        for s, e in make_windows(len(v["ids"]), MAX_LEN, STRIDE):
            lab = v["tag_ids"][s:e] if with_labels else [-100] * (e - s)
            ex.append({"input_ids": v["ids"][s:e], "labels": lab})
    return ex

@torch.no_grad()
def teacher_soft(teachers, examples, pad_id, batch_size=4):
    for t in teachers:
        t.to(device).eval()
    amp = device.type == "cuda"
    out = []
    for i in range(0, len(examples), batch_size):
        chunk = examples[i:i + batch_size]
        b = collate_ner([{"input_ids": e["input_ids"], "labels": [0] * len(e["input_ids"])} for e in chunk], pad_id)
        x = b["input_ids"].to(device)
        acc = 0
        for t in teachers:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                lg = t(x)
            acc = acc + F.log_softmax(lg.float(), dim=-1)
        lp = (acc / len(teachers)).cpu().numpy().astype(np.float16)
        out += [lp[j, :len(e["input_ids"])] for j, e in enumerate(chunk)]
    return out

def collate_kd(batch, pad_id, ignore_index=-100):
    L, C = max(len(b["input_ids"]) for b in batch), batch[0]["soft"].shape[1]
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    lab = torch.full((len(batch), L), ignore_index, dtype=torch.long)
    soft = torch.zeros((len(batch), L, C))
    smask = torch.zeros((len(batch), L), dtype=torch.bool)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        lab[i, :n] = torch.tensor(b["labels"])
        soft[i, :n] = torch.from_numpy(b["soft"].astype(np.float32))
        smask[i, :n] = True
    return {"input_ids": ids, "labels": lab, "soft": soft, "soft_mask": smask}

def train_ner_kd(ner_model, examples, pad_id, cfg, device, val_fn=None, save_path=None, T=2.0, kd_alpha=0.5):
    ner_model.to(device)
    head_ids = {id(p) for p in ner_model.classifier.parameters()}
    head_params = [p for p in ner_model.parameters() if id(p) in head_ids and p.requires_grad]
    backbone_params = [p for p in ner_model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

    loader = DataLoader(examples, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=lambda b: collate_kd(b, pad_id))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    best, history = -1.0, []
    for epoch in range(cfg["epochs"]):
        ner_model.train()
        run = {"loss": 0.0, "ce": 0.0, "kd": 0.0}
        n_batches = 0
        opt.zero_grad()
        for i, b in enumerate(loader):
            x, y = b["input_ids"].to(device), b["labels"].to(device)
            soft, smask = b["soft"].to(device), b["soft_mask"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = ner_model(x)
            logits = logits.float()
            C = logits.size(-1)
            n_hard = (y != -100).sum().clamp(min=1)
            ce = F.cross_entropy(logits.view(-1, C), y.view(-1), ignore_index=-100, reduction="sum") / n_hard
            s_logp = F.log_softmax(logits / T, dim=-1)
            t_prob = F.softmax(soft / T, dim=-1)                
            kl = (t_prob * (torch.log(t_prob.clamp_min(1e-9)) - s_logp)).sum(-1)
            kd = (kl * smask).sum() / smask.sum().clamp(min=1) * (T * T)
            loss = kd_alpha * ce + (1 - kd_alpha) * kd
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run["loss"] += loss.item(); run["ce"] += ce.item(); run["kd"] += kd.item()
            n_batches += 1
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(ner_model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad()
        rec = {"epoch": epoch + 1, **{k: v / max(1, n_batches) for k, v in run.items()}}
        if val_fn is not None:
            rec["val"] = val_fn(ner_model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(ner_model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        ner_model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val={best:.4f})")
    return history


DECODE_KW = dict(collapse=True, gap=1, o_bias=0.5, snap=True)
CFG_KD = {**CFG_LORA, "lr_head": 1e-3, "weight_decay": 0.01}

def make_val_fn(decode_kw=None):
    decode_kw = decode_kw or DECODE_KW

    def val_fn(m):
        merged = get_merged_scores(m, tok2, PAD_ID, MAX_LEN, STRIDE, device)
        pr = decode(merged, texts2, tok2, tags, labels, **decode_kw)
        return strict_scores(gold2, pr, labels)["macro"]["f1"]
    return val_fn

def run_kd(model_pt, teachers, tag="kd", T=2.0, kd_alpha=0.5, r=32, lora_alpha=64,target_modules=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),student="lora", extra_unlabeled=None, cfg=None):
    ex = windows_from_tok(tok1, with_labels=True)
    print(f"labelled windows: {len(ex)}")
    if extra_unlabeled is not None:
        extra = windows_from_tok(extra_unlabeled, with_labels=False)
        print(f"unlabelled windows (soft targets only): {len(extra)}")
        ex = ex + extra
    softs = teacher_soft(teachers, ex, PAD_ID)
    kd_examples = [{"input_ids": e["input_ids"], "labels": e["labels"], "soft": s} for e, s in zip(ex, softs)]

    bb = load_phase1_weights(build_model(), model_pt)
    ner = NERModel(bb, len(tags), head_path=HEAD_PATH)
    if student == "lora":
        ner = apply_lora(ner, r, lora_alpha, 0.05, target_modules)
    else:
        set_trainable(ner, "full")
    hist = train_ner_kd(ner, kd_examples, PAD_ID, cfg or CFG_KD, device, val_fn=make_val_fn(),save_path=f"/kaggle/working/ner_{tag}_best.pt", T=T, kd_alpha=kd_alpha)
    return ner, hist

def report_kd(student, teachers):
    f1 = lambda merged: strict_scores(gold2, decode(merged, texts2, tok2, tags, labels, **DECODE_KW), labels)["macro"]["f1"]
    ms = [get_merged_scores(t, tok2, PAD_ID, MAX_LEN, STRIDE, device) for t in teachers]
    for i, m in enumerate(ms):
        print(f"teacher {i}: {f1(m):.4f}")
    ens = {d: np.mean([m[d] for m in ms], axis=0) for d in ms[0]}
    print(f"teacher ensemble: {f1(ens):.4f}")
    print(f"student: {f1(get_merged_scores(student, tok2, PAD_ID, MAX_LEN, STRIDE, device)):.4f}")

def write_kd_submission(student, decode_kw=None, json_path="prediction.json", zip_path="submission_kd.zip"):
    merged = get_merged_scores(student, tok3, PAD_ID, MAX_LEN, STRIDE, device)
    preds = decode(merged, texts3, tok3, tags, labels, **(decode_kw or DECODE_KW))
    write_submission(preds, json_path, zip_path)
    return preds

In [10]:
hf_dataset_repo = "Exploration-Lab/CS-781-Capstone"
P2 = "Phase_2_LNER"
p2_files = {
    "fold1_text": f"{P2}/Train/fold1-text.json", "fold1_gold": f"{P2}/Train/fold1-gold.json",
    "fold2_text": f"{P2}/Train/fold2-text.json", "fold2_gold": f"{P2}/Train/fold2-gold.json",
    "labels": f"{P2}/Train/labels.txt",          "fold3_text": f"{P2}/Test/fold3-text.json",
}
p2 = {k: hf_hub_download(repo_id=hf_dataset_repo, filename=f, repo_type="dataset", token=HF_TOKEN)
      for k, f in p2_files.items()}


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

In [ ]:
MAX_LEN, STRIDE = 1024, 768     
load = lambda k: json.load(open(p2[k], encoding="utf-8"))
texts1, texts2, texts3 = load("fold1_text"), load("fold2_text"), load("fold3_text")
raw1, raw2 = load("fold1_gold"), load("fold2_gold")
labels = [l.strip() for l in open(p2["labels"], encoding="utf-8") if l.strip()]
print(len(labels), "labels:", labels)
k = next(iter(raw1)); print("gold format check:", k, raw1[k][:2])   
gold1, _ = normalize_gold(raw1, texts1)
gold2, _ = normalize_gold(raw2, texts2)
print("overlapping gold spans:", find_overlaps(gold1), find_overlaps(gold2))


In [ ]:
bpe_data = pickle.load(open("/kaggle/working/legal_bpe_tokenizer.pkl", "rb"))
VOCAB_SIZE=bpe_data["vocab_size"]
PAD_ID = bpe_data["stoi"]["<pad>"]
encode_off = make_offset_encoder(bpe_data, pre_tokenize_preserve, word_to_symbols,apply_bpe_merges, WORD_END)

In [ ]:
tags, tag2id = build_tag_maps(labels)
def tokenize_docs(texts, gold=None):
    out = {}
    for d, t in texts.items():
        ids, offs = encode_off(t)
        tid = [tag2id[x] for x in spans_to_token_tags(offs, gold[d])] if gold else None
        out[d] = {"ids": ids, "offs": offs, "tag_ids": tid}
    return out
MAX_LEN, STRIDE = 1024, 768         
tok1, tok2, tok3 = tokenize_docs(texts1, gold1), tokenize_docs(texts2, gold2), tokenize_docs(texts3)
train_ex = [e for d, v in tok1.items()
            for e in make_training_examples(d, v["ids"], v["tag_ids"], MAX_LEN, STRIDE)]
print("train docs:", len(tok1), "| windows:", len(train_ex),
      "| avg tokens/doc:", sum(len(v["ids"]) for v in tok1.values()) // len(tok1))

In [ ]:
def _train(ner, cfg, tag):
    def val_fn(m):
        preds = predict_docs(m, texts2, tok2, tags, PAD_ID, MAX_LEN, STRIDE, device)
        return strict_scores(gold2, preds, labels)["macro"]["f1"]
    hist = train_ner(ner, train_ex, PAD_ID, cfg, device, val_fn=val_fn,
                     save_path=f"/kaggle/working/ner_{tag}_best.pt")
    return ner, hist

def run_ft(model_pt, mode, n_last_layers=4, tag="exp", cfg=None):
    bb = load_phase1_weights(build_model(), model_pt)
    ner = NERModel(bb, len(tags), head_path=HEAD_PATH)
    set_trainable(ner, mode, n_last_layers, LAYERS_PATH, FINAL_NORM)
    return _train(ner, cfg or CFG, tag)

def run_lora(model_pt, tag="exp", r=8, alpha=16, dropout=0.1,
             target_modules=("q_proj", "v_proj"), cfg=None):
    bb = load_phase1_weights(build_model(), model_pt)
    ner = NERModel(bb, len(tags), head_path=HEAD_PATH)
    ner = apply_lora(ner, r, alpha, dropout, target_modules)
    return _train(ner, cfg or CFG_LORA, tag)

def _make_ner(model_pt):
    bb = load_phase1_weights(build_model(), model_pt)
    return NERModel(bb, len(tags), head_path=HEAD_PATH)

def run_lorma(model_pt, tag="lorma", r=8, alpha=16, variant="plus",
              target_modules=("q_proj", "v_proj"), dropout=0.1, cfg=None):
    ner = apply_lorma(_make_ner(model_pt), r, alpha, variant, dropout, target_modules)
    return _train(ner, cfg or CFG_LORMA, tag)

def run_lora_bilstm(model_pt, tag="lora_bi", cfg=None, lstm_hidden=256,lstm_layers=1, r=32, alpha=64, dropout=0.1,
                    target_modules=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc")):
    bb = load_phase1_weights(build_model(), model_pt)
    ner = NERModelBi(bb, len(tags), PAD_ID, head_path=HEAD_PATH, lstm_hidden=lstm_hidden,lstm_layers=lstm_layers)
    ner = apply_lora(ner, r, alpha, dropout, target_modules)    
    return _train(ner, cfg or CFG_LORA, tag)                     

def run_lora_v2(model_pt, tag="lora_v2.1", cfg=None, **kw):
    bb = load_phase1_weights(build_model(), model_pt)
    ner = NERModel(bb, len(tags), head_path=HEAD_PATH)
    ner = apply_lora_v2(ner, **kw)
    return _train_v2(ner, cfg or CFG_LORA, tag)


In [ ]:
CFG_FT = {**CFG, "epochs": 15, "lr_head": 1e-3}
ner_full, hist_full = run_ft(MODEL_PT, "full", tag="full_fine_tuning",cfg=CFG_FT)

In [ ]:
CFG_FT = {**CFG, "epochs": 15, "lr_head": 1e-3}
ner_part, hist_part = run_ft(MODEL_PT, "partial", n_last_layers=4, tag="partial_fine_tuning",cfg=CFG_FT)

In [ ]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 20.9 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
CFG = dict(epochs=15, batch_size=2, grad_accum=8, lr_backbone=1e-4, lr_head=1e-3,weight_decay=0.01, warmup_ratio=0.1, max_grad_norm=1.0, fp16=True)
CFG_LORA = {**CFG, "lr_backbone": 5e-4}

In [ ]:
ner_lora_v2, hist_lora_v2=run_lora(MODEL_PT,tag="lora",r=16,alpha=32,target_modules=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"))

In [ ]:
ner_lora_v32, hist_lora_v32=run_lora(MODEL_PT,tag="lora",r=32,alpha=64,target_modules=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"))

In [ ]:
ner_lora_v64, hist_lora_v64=run_lora(MODEL_PT,tag="lora",r=64,alpha=128,target_modules=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"))

In [ ]:
CFG_L2 = {**CFG_LORA, "lr_head": 1e-3, "weight_decay": 0.01}
ner_a, h_a = run_lora_v2(MODEL_PT, tag="lora_v32_2", cfg=CFG_L2)     

In [ ]:
ner_bi,h_bi = run_lora_bilstm(MODEL_PT, tag="lora_bi", cfg=CFG_L2)

In [ ]:
ner_bi_v2,h_bi_v2 = run_lora_bilstm(MODEL_PT, tag="lora_bi_h512",lstm_hidden=512)

In [ ]:
CFG_LORMA = {**CFG_LORA, "epochs": 15}  

In [ ]:
ner_lq, h_lq = run_lorma(MODEL_PT, tag="lorma_pi", r=16, alpha=32, variant="pi")

In [ ]:
teachers = [ner_lora_v32, ner_bi]        
student, h_kd = run_kd(MODEL_PT, teachers, tag="kd", r=32, lora_alpha=64)   
report_kd(student, teachers)                   
preds = write_kd_submission(student)           

In [ ]:
preds = predict_docs(ner_bi_v2, texts3, tok3, tags, PAD_ID, MAX_LEN, STRIDE, device)
write_submission(preds, "prediction.json", "submission_lora_v32_bi_v2.zip")